# 🐾 Animal Sound Generator — v14 Latent Diffusion

**True generation from noise.** VAE compression → latent diffusion → Griffin-Lim audio.

| Phase | Time (L4) |
|-------|:---------:|
| Setup | ~2 min |
| VAE | ~30 min |
| Latent Diffusion | ~20 min |
| Generate | ~2 min |

**Fixes v1-v13 failures:**
- VAE decoder WITHOUT encoder skips (self-attention instead)
- Diffusion on 256-dim latent (not 35K mel bins)
- Griffin-Lim audio (mathematical, no training data mismatch)

### Before running:
1. Runtime → L4 GPU
2. Push your code to GitHub first

In [ ]:
# @title 1. Setup
!git clone https://github.com/thanhbm-94/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib tqdm soundfile

!mkdir -p models

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title 2. Download ESC-50 Data (640 clean animal sound clips)
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /tmp/esc50.zip
!unzip -qo /tmp/esc50.zip -d /tmp/
!python src/scripts/setup_esc50.py --source /tmp/ESC-50-master/audio --target data/esc50

!ls data/esc50/

In [ ]:
# @title 3. Train VAE (Phase 1 — compression)
!python src/train_v14.py --phase 1 --mode train

In [ ]:
# @title 4. Train Latent Diffusion (Phase 2 — generation)
!python src/train_v14.py --phase 2 --mode train

In [ ]:
# @title 5. Generate all 7 animal sounds
!python src/generate.py --v14-latent --count 1 --output-dir outputs

import os
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}') / 1024
    print(f'  {f} ({size:.0f} KB)')

In [ ]:
# @title 6. Download generated sounds
import zipfile, os
with zipfile.ZipFile('v14_animal_sounds.zip', 'w') as z:
    for f in os.listdir('outputs'):
        if f.endswith('.wav'): z.write(f'outputs/{f}', f)
from google.colab import files
files.download('v14_animal_sounds.zip')